In [ ]:

import os 
import numpy as np 
import json
from itertools import islice
import scipy.stats as stats
import copy
import matplotlib.pyplot as plt
import natural_quantization.analysis as nqa 
from natural_quantization.preprocess import one_hot
from natural_quantization.analysis import binomial_error, mode_of_nested_list, concat_positionwise

working_dir = "/Users/dlakhdar/physics/copy_repos/natural-quantization/data"
experiment_dir = "experiment_data/statistics/"
index_file = "2025-05-28_19:22:50.245849_indices.txt"
index_file_path = os.path.join(working_dir,experiment_dir,index_file)
As = [0.0,.1,.2,.5,1.0,1.5,2.0]
num_as = len(As)

with open(f"{working_dir}/mnist_data/mnist_data_reduced.json") as image_file:
    image_data = json.load(image_file)

test_set = image_data["test_set"]
# print(f"data:\n {test_set[0][0]}")
x_test, y_test = ([np.array(d[0]) for d in test_set], 
                  [one_hot(d[1]) for d in test_set])


begin_run = 1
end_run = 5 

def prepare_data_statistics(begin_run,end_run,working_dir,experiment_dir,num_as):

    total = []
    indexes = []
    for i in range(begin_run,end_run):
        files = os.listdir(os.path.join(working_dir,experiment_dir,f"run{i}"))
        files = [os.path.join(working_dir,experiment_dir,f"run{i}", file) for file in files]
        files.sort()
        n_samples = len(files)-1  # -1 for the indices file
        data = []

        for file in files:
            # print(f"Processing file: {file}")
            # 1) Read the file
            if "unknownindices" in file : 
                continue

            else: 
                with open(file, "r") as f:
                    text = f.read() 
                    # print(f"Text: {text[:100]}...")  # Print the first 100 characters for debugging
                
                if "indices" in file:
                    indices = eval(
                                text,
                                {"__builtins__": None},   # disable built-ins for safety
                            ) 
                        
                    indices= indices[0:n_samples]
                    indexes.append(indices)
                    print(f"Indices: {indices}")

                else: 
                    
                    if 'np.int64' in text:
                        tmp=eval(
                            text,
                            # {"__builtins__": None},   # disable built-ins for safety
                            # {"np.int64": np.int64}               # map the name 'np' to numpy
                        )
                    elif 'array' in text:   
                        tmp = eval(
                                text,
                                {"__builtins__": None},   # disable built-ins for safety
                                {"array": np.array}     # map the name 'array' to numpy's array constructor
                            )
                    
                    # print(f"Data: {tmp}")
                    
                    data= data[0:n_samples]
                    data.append(tmp)

        data = sum(data, []) 

        for image in data:
            concatenated = np.concatenate(image[-1]).flatten()
            image[-1] = concatenated
        
        data = [data[i:num_as+i] for i in range(0,len(data),num_as)]

        new_data = [[] for _ in range(num_as)]
        for d in data:
            for i in range(0,len(d)):
                new_data[i].append(d[i][-1])

        new_data_copy = copy.copy(new_data)
        for i in range(num_as):
            for j in range(n_samples):
                new_data_copy[i] = mode_of_nested_list(new_data[i])

        
        total.append(new_data_copy)

    predicted_labels = concat_positionwise(total)
    indexes = sum(indexes, []) 
    
    return predicted_labels, indexes

In [167]:
predicted_labels,indexes = prepare_data_statistics(begin_run,end_run,working_dir,experiment_dir+"quantum",num_as)

simulated_predicted_labels,simulated_indexes = prepare_data_statistics(1,2,working_dir,experiment_dir+"simulation",num_as)


Indices: [8584, 6230, 9036, 999, 1736, 241, 3846, 2749, 2110, 9028, 7207, 9609, 6354, 8871, 4646, 9519, 1909, 6490, 8460, 2215]
Indices: [2865]
Indices: [9979, 4089, 8303, 2182]
Indices: [27, 9728, 4126, 9511, 8295, 1799, 440, 4110, 744, 1838, 3633, 2016, 7125, 8796, 4033, 6976, 8593, 7136, 1486, 6468, 4743, 9027, 3430, 94, 481, 536, 8545, 2136, 6197, 1310]
Indices: [2614, 874, 4283, 9669, 6603, 4487, 2505, 5023, 4734, 3270, 1378, 2351, 3935, 2623, 473, 2763, 1153, 700, 5998, 1529, 8190, 4007, 4556, 8556, 170, 5137, 3023, 4627, 1383, 5776, 311, 3812, 1727, 1118, 7646, 3822, 5041, 8066, 4137, 3306, 3140, 4054, 6946, 9243, 1000, 4540, 7470, 171, 342, 9540, 6950, 1897, 6416, 1956, 9110]


In [213]:
def produce_accuracy_error(indexes,predicted_labels,y_test,n_samples):
    accuracies = []
    true_labels = np.array([y_test[i] for i in indexes ])
    true_labels = np.argmax(true_labels,axis=1)
    for i in range(num_as):
        accuracies.append(np.mean(true_labels == predicted_labels[i]))

    errors=binomial_error(np.array(accuracies),n_samples)
    return accuracies, errors

accuracy, error = produce_accuracy_error(indexes,predicted_labels,y_test,10)
simulated_accuracy, simulated_error = produce_accuracy_error(simulated_indexes,simulated_predicted_labels,
                                                             y_test,10)

M=55
error = (1/M) * np.sqrt(np.array(accuracy)**2)
simulated_error = (1/M) * np.sqrt(np.array(simulated_accuracy)**2)

In [210]:

plt.rcParams.update({
    "mathtext.fontset":  "cm",               # use Computer Modern for math
    "axes.labelsize":    14,
    "axes.titlesize":    16,
    "legend.fontsize":   12,
    "xtick.labelsize":   12,
    "ytick.labelsize":   12,
    "axes.grid":         True,
    "grid.linestyle":    "--",
    "grid.alpha":        0.4,
})


# plt.figure(figsize=(15,8))
# plt.title("Validation Rate vrs. Quatumness: 16 Qubits, 3 Layers, 10 Samples per a")
# plt.plot(As,accuracy,color="black",label="Quantum Device")
# plt.plot(As,simulated_accuracy,'x--',color="black",label="Simulation")


# plt.scatter(As,accuracies
# ,color="orange")
plt.figure(figsize=(8,6))
plt.errorbar(x=As,y=accuracy,yerr=error,fmt='o-',capsize=4,barsabove=False, 
label=r"Superconducting Device",color="blue",ecolor="blue",elinewidth=1.5)
plt.errorbar(x=As,y=simulated_accuracy,yerr=simulated_error,fmt='x--',capsize=4,
             barsabove=False, label=r"Simulation",color="orange",ecolor="orange",alpha=.9,elinewidth=1.5)
plt.xlabel(r"Quantumness ($a$)")
plt.legend()
plt.grid()
plt.ylabel("Validation rate")
plt.xticks(As)
plt.yticks(np.arange(.6,1.05,.1))
plt.margins(x=0.05)
plt.ylim(.6, 1.00)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

/var/folders/04/2cqfhcv133s3gbkf3b35tnkr0000gn/T/ipykernel_32400/2641076030.py:37: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [131]:
working_dir = "/Users/dlakhdar/physics/copy_repos/natural-quantization/data"
experiment_dir = "experiment_data/statistics/simulation"
index_file = "2025-06-12_21:23:23.359927_indices.txt"
index_file_path = os.path.join(working_dir,experiment_dir,index_file)
As = [0.0,.1,.2,.5,1.0,1.5,2.0]
num_as = len(As)


begin_run = 1
end_run = 1

total = []
indexes = []
for i in range(begin_run,end_run):
    files = os.listdir(os.path.join(working_dir,experiment_dir,f"run{i}"))
    files = [os.path.join(working_dir,experiment_dir,f"run{i}", file) for file in files]
    files.sort()
    n_samples = len(files)-1  # -1 for the indices file
    data = []

    for file in files:
        # 1) Read the file
        if "unknownindices" in file : 
            continue

        else: 
            with open(file, "r") as f:
                text = f.read() 
            if "indices" in file:
                indices = eval(
                            text,
                            {"__builtins__": None},   # disable built-ins for safety
                        ) 
                    
                indices= indices[0:n_samples]
                indexes.append(indices)

            else: 
           
                tmp = eval(
                        text,
                        {"__builtins__": None},   # disable built-ins for safety
                        {"array": np.array}       # map the name 'array' to numpy's array constructor
                    )
                
                data= data[0:n_samples]
                data.append(tmp)

    data = sum(data, []) 

    for image in data:
        concatenated = np.concatenate(image[-1]).flatten()
        image[-1] = concatenated
    
    data = [data[i:num_as+i] for i in range(0,len(data),num_as)]

    new_data = [[] for _ in range(num_as)]
    for d in data:
        for i in range(0,len(d)):
            new_data[i].append(d[i][-1])

    new_data_copy = copy.copy(new_data)
    for i in range(num_as):
        for j in range(n_samples):
            new_data_copy[i] = mode_of_nested_list(new_data[i])

    
    total.append(new_data_copy)

predicted_labels = concat_positionwise(total)
indexes = sum(indexes, []) 


In [16]:
file2 = "/Users/dlakhdar/physics/copy_repos/natural-quantization/data/experiment_data/statistics/run2/2025-05-30_17:27:19.700428_indices.txt"

with open(file2,"r") as index_file:
    text = index_file.read()
    indices2 = eval(
            text,
            {"__builtins__": None})  # disable built-ins for safet


indices2 = indices[0]

indices2

8584

In [18]:
file3 = "/Users/dlakhdar/physics/copy_repos/natural-quantization/data/experiment_data/statistics/run2/2025-05-30_17:27:19.700428_indices.txt"

with open(file3,"r") as index_file:
    text = index_file.read()
    indices2 = eval(
            text,
            {"__builtins__": None})  # disable built-ins for safet


indices3 = indices[0:4]
indices3

[8584, 6230, 9036, 999]

In [271]:
fig, ax = plt.subplots(figsize=(8,6))
# Plot central lines
ax.errorbar(As, simulated_accuracy,fmt='o-', yerr=simulated_error, barsabove=True,capsize=5,color="coral", label="QASM Simulation")
ax.errorbar(As, accuracy, fmt='o-', yerr=error, color="purple",barsabove=True, capsize=5,label="Superconducting Device")
# Shaded error regions

# ax.fill_between(As,
#                 np.array(simulated_accuracy) - np.array(simulated_error),
#                 np.array(simulated_accuracy) + np.array(simulated_error),
#                 color="coral", alpha=0.2)

# ax.fill_between(As,
#                 np.array(accuracy) - np.array(error),
#                 np.array(accuracy) + np.array(error),
#                 color="purple", alpha=0.2)
ax.set_xlabel(r"Quantumness ($a$)",fontsize=22)
ax.set_ylabel(r"Validation rate",fontsize=22)
ax.set_xticks(As,)
ax.set_yticks(np.arange(.7,1.00,.05))
# Tick labels
ax.tick_params(axis="both", which="major", labelsize=14)
ax.tick_params(axis="both", which="minor", labelsize=12)
ax.margins(x=0.05)
ax.legend(frameon=False, fontsize=16)
ax.grid(False)
plt.tight_layout()
plt.show()



/var/folders/04/2cqfhcv133s3gbkf3b35tnkr0000gn/T/ipykernel_32400/1506882862.py:27: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [236]:
plt.switch_backend("pgf")

# 2) Tell PGF to use your LaTeX installation
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",   # or "xelatex", "lualatex"
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,          # don’t override your document’s fonts
})

In [272]:
plt.savefig("statistics.pdf")


In [245]:
plt.savefig("statisticsv2.pdf",bbox_inches='tight', pad_inches=0.1)
